In [1]:
import time
import functools 
from dataclasses import dataclass
import sys

import numpy as np
import pandas as pd
import joblib
import bct
import colorcet

import nibabel as nib
from nilearn import masking

import matplotlib.pyplot as plt
from matplotlib import colors
import seaborn as sns

import pymcm

sys.path.append('/RAID1/jupytertmp/mcm/src')
import filenames

%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2

In [2]:
rootdir = '/RAID1/jupytertmp/mcm'
subjects = [3, 7, 12, 14, 17, 20, 23, 25, 26, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38]
cohort2017 = subjects[:9]
cohort2021 = subjects[9:]

# Schaefer region labels

In [3]:
table = pd.read_csv(
    '/RAID1/jupytertmp/mcm/data/external/schaefer/Schaefer2018_400Parcels_7Networks_order.lut', 
    sep=" ", header=None, names=["roi", "r", "g", "b", "label"])

table['label'] = table['label'].apply(lambda x: x.strip())
table = table.assign(
    hemi=table['label'].apply(lambda x: x.split('_')[1]),
    net=table['label'].apply(lambda x: x.split('_')[2]),
    region=table['label'].apply(
        lambda x: x.split('_')[3] if len(x.split('_')) == 5 else x.split('_')[2]
    ),
    component=table['label'].apply(lambda x: x.split('_')[-1]),
)
table = table.assign(hemi_net=table.hemi + ' ' + table.net)

# Native space with Schaefer parcellation 

## Load nifti images

In [4]:
group = pymcm.data.Group(name='Native space analysis with Schaefer atlas')

for subject in subjects:
    
    fmri = pymcm.data.ImagingSpace(name='functional 3mm')
    fmri.add_raws(
        img=nib.load(filenames.bold.format(id=subject)),
        mask=nib.load(filenames.gm_mask_func.format(id=subject)),
        snr_mask=nib.load(filenames.bold_snr_mask.format(id=subject)),
        atlas=nib.load(filenames.schaefer_func.format(id=subject)),
    )

    pet = pymcm.data.ImagingSpace(name='pet 3mm')
    pet.add_raws(
        img=nib.load(filenames.cmrglc_3mm.format(id=subject)),
        mask=nib.load(filenames.gm_mask_pet.format(id=subject)),
        atlas=nib.load(filenames.schaefer_pet.format(id=subject)),
    )

    dwi = pymcm.data.ImagingSpace(name='diffusion 2mm')
    dwi.add_raws(
        img=nib.load(filenames.b0.format(id=subject)),
        mask=nib.load(filenames.gm_mask_dwi.format(id=subject)),
        atlas=nib.load(filenames.schaefer_dwi.format(id=subject))
    )

    sub = pymcm.data.Subject(id=subject, name=f'sub{subject:03}')
    sub.add_raws(
        fmri=fmri, 
        pet=pet,
        dwi=dwi
        )
    
    group.add_raw(sub.name, sub)

In [5]:
group.add_derivs(
    table=table,
    subjects=subjects,
    cohort2017=cohort2017,
    cohort2021=cohort2021,
    rootdir=rootdir
)

## Voxel overlap  and masking

In [6]:
for sub in group.iterraw():
    
    # PET
    intersection = pymcm.processing.voxel_intersection(sub.pet.img, sub.pet.atlas, sub.pet.mask)
    data, labels = masking.apply_mask([sub.pet.img, sub.pet.atlas], intersection)
    sub.pet.add_derivs(
        voxint_mask=intersection,
        masked_data=data,
        masked_labels=labels
    )

    # fMRI
    intersection = pymcm.processing.voxel_intersection(sub.fmri.img, sub.fmri.atlas, sub.fmri.mask, sub.fmri.snr_mask)
    data = masking.apply_mask(sub.fmri.img, intersection)
    labels = masking.apply_mask(sub.fmri.atlas, intersection)
    sub.fmri.add_derivs(
        voxint_mask=intersection,
        masked_data=data, 
        masked_labels=labels
    )
    
    print(sub.name, end='\r')

## Select regions with good data

based on criteria:  
- region has at least 5 voxels in both fmri and pet
- region is present in all subjects
- region is not in Limbic system
    - only 6 regions are in the Limbic system because of the fmri tSNR constraint

In [7]:
nvox_min = 5
regions = []
for sub in group.iterraw():

    # func
    rois, counts = np.unique(sub.fmri['masked_labels'], return_counts=True)
    func_pass = rois[counts >= nvox_min]
    regions.append(func_pass)

    # pet
    rois, counts = np.unique(sub.pet['masked_labels'], return_counts=True)
    pet_pass = rois[counts >= nvox_min]
    regions.append(pet_pass)

    print(sub.name, func_pass.size, pet_pass.size)

sub003 391 399
sub007 392 399
sub012 390 400
sub014 373 400
sub017 388 400
sub020 383 400
sub023 392 400
sub025 394 400
sub026 389 400
sub028 393 400
sub029 393 400
sub030 389 400
sub031 387 400
sub032 396 400
sub033 388 400
sub034 394 400
sub035 395 400
sub036 388 400
sub037 392 400
sub038 394 400


In [20]:
good_rois = table[['roi', 'net']]
common_regions = functools.reduce(np.intersect1d, regions, np.arange(1,401))
good_rois = good_rois.merge(right=pd.DataFrame(dict(regions=common_regions)), how='right', left_on='roi', right_on='regions')
good_rois = good_rois[good_rois.net != 'Limbic']

group.add_derivs(
    roi_template=good_rois['roi'].to_numpy(),
    net_template=good_rois['net'].to_numpy(),
)

print(good_rois.shape)

(362, 3)


## Select data and labels

In [9]:
for sub in group.iterraw():
    for img in sub.iterraw():

        if 'diffusion' in img.name:
            continue

        data, labels = img['masked_data'], img['masked_labels']
        mask = np.isin(labels, group['roi_template'])
        selected_data = data[..., mask] 
        selected_labels = labels[mask]
        img.add_derivs(
            selected_data=selected_data, 
            selected_labels=selected_labels
        )

## ROI statistics

Calculate roi averages, stds and number of observations 

In [10]:
for sub in group.iterraw():
    for img in sub.iterraw():

        if 'diffusion' in img.name:
            continue
        
        data, labels = img['selected_data'], img['selected_labels']
        roiavgs = pymcm.processing.roi_stat(data=data, labels=labels, func='mean')
        img.add_derivs(roiavgs=roiavgs)

## FC and pFC

In [ ]:
for sub in group.iterraw():
    
    timeseries = sub.fmri['roiavgs']
    fc = pymcm.processing.connectivity(timeseries, func=pymcm.array.corr, zero_diag=True, positive=True)
    pfc = pymcm.processing.connectivity(timeseries, func=pymcm.array.partial_corr, zero_diag=True, positive=True)
    degree = fc.sum(axis=0)

    sub.fmri.add_derivs(fc=fc, pfc=pfc, degree=degree)
    print(sub.name, end='\r')

## SC

In [12]:
for sub in group.iterraw():

    sc_filename = filenames.connectome_sift2.format(id=sub.id)

    sc_allrois = pd.read_csv(sc_filename, header=None)
    idcs = group['roi_template'] - 1

    sc_weight = sc_allrois.iloc[idcs, idcs].to_numpy()
    sc_bin = sc_weight > 0
    sc_thr = bct.threshold_proportional(sc_weight, 0.2)
    sc_thr = sc_thr > 0
    degree = sc_weight.sum(axis=0)

    sub.dwi.add_derivs(sc_weight=sc_weight, sc_bin=sc_bin, sc_thr=sc_thr, degree=degree)

### Null models for FC and pFC

needs more work

## PET logratio

In [13]:
for sub in group.iterraw():

    degree = sub.fmri['degree']

    cmrglc_adj = sub.pet['roiavgs'] / degree # adjusted roi averages of cmrglc 
    logratio = pymcm.processing.pet_logratio(cmrglc_adj)
    dir_weights, undir_weights = pymcm.processing.logratio_to_weights(logratio)

    sub.pet.add_derivs(
        cmrglc_adj=cmrglc_adj,
        logratio=logratio, 
        dir_weights=dir_weights, 
        undir_weights=undir_weights
        )

## Directed and undirected connectivity

In [14]:
for sub in group.iterraw():

    fc = sub.fmri['fc']
    sc = sub.dwi['sc_thr']

    dir_weight = sub.pet['dir_weights']
    undir_weight = sub.pet['undir_weights']

    connectivity = sc * fc
    # connectivity = sub.dwi['sc_weight']
    directed = connectivity * dir_weight
    undirected = connectivity * undir_weight

    assert np.allclose(connectivity, undirected + directed + directed.T)

    sub.add_derivs(
        connectivity=connectivity,
        directed=directed, 
        undirected=undirected
    )

## Network averages

In [15]:
labels = group['net_template']
for sub in group.iterraw():
    netstats = pymcm.processing.matrix_label_aggregate(sub['directed'], labels)
    sub.add_derivs(netstats=netstats)

# Calculate data tables

## Group averages

In [16]:
group.add_derivs(
    fc =            np.mean([sub.fmri['fc'] for sub in group.iterraw()], axis=0),
    pfc =           np.mean([sub.fmri['pfc'] for sub in group.iterraw()], axis=0),
    sc_thr =        np.mean([sub.dwi['sc_thr'] for sub in group.iterraw()], axis=0),
    sc_weight =     np.mean([sub.dwi['sc_weight'] for sub in group.iterraw()], axis=0),
    logratio =      np.mean([sub.pet['logratio'] for sub in group.iterraw()], axis=0),
    # 
    connectivity =  np.mean([sub['connectivity'] for sub in group.iterraw()], axis=0),
    directed =      np.mean([sub['directed'] for sub in group.iterraw()], axis=0),
    undirected =    np.mean([sub['undirected'] for sub in group.iterraw()], axis=0),
    # 
    netavg_percent_in =     np.mean([sub['netstats']['degree_in_percent'] for sub in group.iterraw()], axis=0),
    netavg_percent_out =    np.mean([sub['netstats']['degree_out_percent'] for sub in group.iterraw()], axis=0),
    netavg_wrt_outputs =    np.mean([sub['netstats']['avg_outputs'] for sub in group.iterraw()], axis=0),
    netavg_wrt_inputs =     np.mean([sub['netstats']['avg_inputs'] for sub in group.iterraw()], axis=0),
    netavg_wrt_reciprocal = np.mean([sub['netstats']['avg_reciprocal'] for sub in group.iterraw()], axis=0),
    netavg_wrt_mixed =      np.mean([sub['netstats']['avg_mixed'] for sub in group.iterraw()], axis=0),
    netavg_mean =           np.mean([sub['netstats']['avgs'] for sub in group.iterraw()], axis=0),
)

---

# Sanity checks

In [17]:
for sub in group.iterraw():

    assert np.all(np.logical_and(sub.fmri['fc'] >= 0, sub.fmri['fc'] <= 1))
    assert np.all(np.logical_and(sub.fmri['pfc'] >= 0, sub.fmri['pfc'] <= 1))

    assert sub.dwi['sc_bin'].dtype is np.dtype('bool')
    assert sub.dwi['sc_thr'].dtype is np.dtype('bool')

    assert np.allclose(sub['connectivity'], sub['undirected'] + sub['directed'] + sub['directed'].T)    

---

# Visualize the group average matrices

In [ ]:
fig, ax = plt.subplots(figsize=(5,5))
sns.heatmap(group['fc'], square=True, cbar=True, cbar_kws=dict(shrink=0.5), cmap='cet_CET_L1_r', ax=ax)
ax.axis('off')
ax.set_title('FC avg')

fig, ax = plt.subplots(figsize=(5,5))
sns.heatmap(group['pfc'], square=True, cbar=True, cbar_kws=dict(shrink=0.5), cmap='cet_CET_L1_r', ax=ax, norm=colors.PowerNorm(0.5))
ax.axis('off')
ax.set_title('pFC avg')

fig, ax = plt.subplots(figsize=(5,5))
sns.heatmap(group['sc_weight'], square=True, cbar=True, cbar_kws=dict(shrink=0.5), cmap='cet_CET_L1_r', ax=ax, norm=colors.PowerNorm(0.3))
ax.axis('off')
ax.set_title('SC weight avg')

fig, ax = plt.subplots(figsize=(5,5))
sns.heatmap(group['logratio'], square=True, cbar=True, cbar_kws=dict(shrink=0.5), cmap='cet_CET_D1', ax=ax)
ax.axis('off')
ax.set_title('Logratio avg')

fig, ax = plt.subplots(figsize=(5,5))
sns.heatmap(group['directed'], square=True, cbar=True, cbar_kws=dict(shrink=0.5), cmap='cet_CET_L1_r', ax=ax, norm=colors.PowerNorm(0.3))
ax.axis('off')
ax.set_title('Directed avg')

fig, ax = plt.subplots(figsize=(5,5))
sns.heatmap(group['undirected'], square=True, cbar=True, cbar_kws=dict(shrink=0.5), cmap='cet_CET_L1_r', ax=ax, norm=colors.PowerNorm(0.3))
ax.axis('off')
ax.set_title('Unirected avg')

# fig, ax = plt.subplots(figsize=(5,5))
# sns.heatmap(group['net_avg'], square=True, cbar=False, cmap='cet_CET_L18', ax=ax)
# ax.set(xticklabels=np.unique(group['net_template']), yticklabels=np.unique(group['net_template']))
# ax.set_title('Network directed avg')

# Save/load data

## Remove image data from the images

Saves memory and disk space, see [here](https://nipy.org/nibabel/images_and_memory.html)

In [19]:
for sub in group.iterraw():
    for space in sub.iterraw():
        for img in space.iterraw():
            img.uncache() 

## Save

In [ ]:
fname = '/RAID1/jupytertmp/mcm/data/data_group-tum.joblib'

print(f"Saved on {time.strftime('%d %B %Y - %H:%M')}")
_ = joblib.dump(group, fname)

## Load

In [21]:
# fname = '/RAID1/jupytertmp/mcm/data/DATA_FINAL.joblib'
# group = joblib.load(fname)

# Clear up

In [ ]:
%reset -f

import IPython
IPython.Application.instance().kernel.do_shutdown(True)

---